In [ ]:
peg_price_data = pd.read_csv(Path('Panel_A_Data') / 'Secondary_Prices_and_Peg_Deviations_Daily.csv')
redemption_pressure_data = pd.read_csv(Path('Panel_A_Data') / 'Redemption_Pressure_Index_Daily.csv')
SOFR_ = pd.read_csv(Path('Panel_B_Data') / 'SOFR.csv')
SOFR_OIS_1m = pd.read_csv(Path('Panel_B_Data') / 'USD_1M_SOFR_OIS.csv')
SOFR_OIS_3m = pd.read_csv(Path('Panel_B_Data') / 'USD_3M_SOFR_OIS.csv')

# Equation A1: peg shortfall (bps) = max[0, (1 - price in USD) x 10,000]
# Equation A2: redemption pressure (bps) = peg shortfall (bps) + net burn intensity (bps)
# Net burn intensity in the source file was constructed as:
# max(0, -daily supply change) / previous-day circulating supply x 10,000.
constructed_redemption_pressure = redemption_pressure_data[['Date']].copy()
for symbol in ['USDT', 'USDC', 'DAI']:
    price = peg_price_data[f'{symbol}_Price_USD']
    peg_shortfall = np.maximum(0.0, (1.0 - price) * 10_000)
    burn_intensity = redemption_pressure_data[f'{symbol}_Net_Burn_Intensity_bps']
    constructed_redemption_pressure[f'{symbol}_Peg_Shortfall_bps_eq_A1'] = peg_shortfall
    constructed_redemption_pressure[f'{symbol}_Redemption_Pressure_bps_eq_A2'] = (
        peg_shortfall + burn_intensity
    )

# Equations B1/B2: SOFR-OIS spread (bps) = (SOFR rate - OIS rate) x 100
sofr_for_equations = SOFR_.rename(columns={'observation_date': 'Date'}).copy()
ois_1m_for_equations = SOFR_OIS_1m.copy()
ois_3m_for_equations = SOFR_OIS_3m.copy()
for frame in [sofr_for_equations, ois_1m_for_equations, ois_3m_for_equations]:
    frame['Date'] = pd.to_datetime(frame['Date'], utc=True).dt.tz_localize(None).dt.normalize()

constructed_sofr_ois_spreads = (sofr_for_equations[['Date', 'SOFR']].merge(ois_1m_for_equations[['Date', 'USD_1M_SOFR_OIS']], on='Date', how='inner').merge(ois_3m_for_equations[['Date', 'USD_3M_SOFR_OIS']], on='Date', how='inner').sort_values('Date').reset_index(drop=True))
constructed_sofr_ois_spreads['SOFR_minus_1M_OIS_bps_eq_B1'] = (constructed_sofr_ois_spreads['SOFR'].sub(constructed_sofr_ois_spreads['USD_1M_SOFR_OIS']).mul(100))

constructed_sofr_ois_spreads['SOFR_minus_3M_OIS_bps_eq_B2'] = (
    constructed_sofr_ois_spreads['SOFR']
    .sub(constructed_sofr_ois_spreads['USD_3M_SOFR_OIS'])
    .mul(100)
)

# Saved DataFrame variables:
# constructed_redemption_pressure (Equations A1-A2)
# constructed_sofr_ois_spreads (Equations B1-B2)

In [ ]:
# Extract Circle's monthly USDC reserve-composition reports
# Note: Circle has no public historical reserve-composition API. Its official
# transparency page links the monthly assurance PDFs used below.
import re
from io import BytesIO
from urllib.parse import urljoin, unquote
import requests
from bs4 import BeautifulSoup
import pdfplumber

CIRCLE_TRANSPARENCY_URL = 'https://www.circle.com/transparency'
CIRCLE_HEADERS = {'User-Agent': 'Risk-Transmission-Research/1.0'}

response = requests.get(CIRCLE_TRANSPARENCY_URL, headers=CIRCLE_HEADERS, timeout=60)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
report_urls = pd.Series([
    urljoin(CIRCLE_TRANSPARENCY_URL, link['href'])
    for link in soup.select('a[href]')
    if unquote(link['href']).lower().endswith('.pdf')
    and 'usdc' in unquote(link['href']).lower()
], dtype='string').drop_duplicates().tolist()

if not report_urls:
    raise RuntimeError('No USDC monthly report links were returned by Circle.')

month_numbers = {
    month.lower(): number for number, month in enumerate(
        ['January', 'February', 'March', 'April', 'May', 'June',
         'July', 'August', 'September', 'October', 'November', 'December'], 1
    )
}

def month_from_report_url(url):
    decoded = unquote(url)
    year = re.search(r'(?<!\d)(20\d{2})(?!\d)', decoded)
    month = re.search('|'.join(month_numbers), decoded, flags=re.IGNORECASE)
    if year and month:
        return pd.Timestamp(int(year.group()), month_numbers[month.group().lower()], 1)
    return pd.NaT

circle_monthly_report_catalog = pd.DataFrame({'Report_URL': report_urls})
circle_monthly_report_catalog['Report_Month'] = (
    circle_monthly_report_catalog['Report_URL'].map(month_from_report_url)
)
circle_monthly_report_catalog = (
    circle_monthly_report_catalog.dropna(subset=['Report_Month'])
    .sort_values('Report_Month').reset_index(drop=True)
)

# Extract every table row because Circle's PDF layout and category names vary by year.
reserve_rows = []
for report in circle_monthly_report_catalog.itertuples(index=False):
    pdf_response = requests.get(report.Report_URL, headers=CIRCLE_HEADERS, timeout=120)
    pdf_response.raise_for_status()
    with pdfplumber.open(BytesIO(pdf_response.content)) as pdf:
        for page_number, page in enumerate(pdf.pages, 1):
            for table_number, table in enumerate(page.extract_tables(), 1):
                for row_number, cells in enumerate(table, 1):
                    cleaned = [
                        re.sub(r'\s+', ' ', str(cell)).strip() if cell is not None else pd.NA
                        for cell in cells
                    ]
                    reserve_rows.append({
                        'Report_Month': report.Report_Month, 'Page': page_number,
                        'Table': table_number, 'Row': row_number,
                        'Cells': cleaned, 'Report_URL': report.Report_URL,
                    })

circle_monthly_reserve_composition_raw = pd.DataFrame(reserve_rows)
if circle_monthly_reserve_composition_raw.empty:
    raise RuntimeError('Reports downloaded, but no PDF tables were extracted.')

max_cells = circle_monthly_reserve_composition_raw['Cells'].map(len).max()
cell_data = pd.DataFrame(
    circle_monthly_reserve_composition_raw.pop('Cells').tolist(),
    columns=[f'Cell_{i}' for i in range(1, max_cells + 1)],
)
circle_monthly_reserve_composition = pd.concat(
    [circle_monthly_reserve_composition_raw, cell_data], axis=1
)
circle_monthly_reserve_composition.to_csv(
    PANEL_A_DATA_FOLDER / 'Circle_USDC_Monthly_Reserve_Composition_Raw.csv', index=False
)
circle_monthly_report_catalog.to_csv(
    PANEL_A_DATA_FOLDER / 'Circle_USDC_Monthly_Report_Catalog.csv', index=False
)
circle_monthly_reserve_composition.head()